In [8]:
#pip install vl-convert-python

In [9]:
import altair as alt
import pandas as pd

df = pd.read_csv("healthcare_cleaned.csv")

In [10]:
conditions = sorted(df['medical_condition'].dropna().unique().tolist())

custom_colors = [
    "#E63946", "#457B9D", "#2A9D8F", "#E9C46A", "#F4A261", "#264653"
]

providers = sorted(df['insurance_provider'].dropna().unique().tolist())
colors = custom_colors[:len(providers)]
color_scale = alt.Scale(domain=providers, range=colors)

charts = []

for condition in conditions:
    df_cond = df[df['medical_condition'] == condition]

    yearly_avg = (
        df_cond.groupby(['year', 'insurance_provider'])['billing_amount']
        .mean()
        .reset_index()
        .rename(columns={'billing_amount': 'avg_billing'})
    )

    base = alt.Chart(yearly_avg).encode(
        x=alt.X('year:O',
                title='Year',
                axis=alt.Axis(labelAngle=0, tickSize=0, grid=False)),
        y=alt.Y('avg_billing:Q',
                title='Avg Billing ($)',
                scale=alt.Scale(zero=False),
                axis=alt.Axis(gridColor='#e0e0e0', gridDash=[4, 4], tickSize=0)),
        color=alt.Color('insurance_provider:N',
                        title='Insurance Provider',
                        scale=color_scale),
        tooltip=[
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('insurance_provider:N', title='Insurance Provider'),
            alt.Tooltip('avg_billing:Q', title='Avg Billing ($)', format='$,.0f')
        ]
    )

    lines = base.mark_line(strokeWidth=2.5, opacity=0.9)
    points = base.mark_point(size=80, filled=True, opacity=1.0, strokeWidth=1.5, stroke='white')

    chart = (lines + points).properties(
        width=280,
        height=200,
        title=alt.TitleParams(
            text=condition.title(),
            fontSize=13,
            anchor='start'
        )
    )

    charts.append(chart)

# Arrange into a panel — 3 columns
panel = alt.vconcat(*[
    alt.hconcat(*charts[i:i+3])
    for i in range(0, len(charts), 3)
]).properties(
    title=alt.TitleParams(
        text="Average Billing Over Time by Medical Condition",
        subtitle="Grouped by insurance provider | 2019–2024",
        fontSize=18,
        subtitleFontSize=13,
        subtitleColor='#666666',
        anchor='start',
        offset=15
    )
).configure_axis(
    labelFontSize=11,
    titleFontSize=12,
    labelColor='#444444',
    titleColor='#222222'
).configure_legend(
    labelFontSize=11,
    titleFontSize=12,
    symbolSize=100,
    symbolStrokeWidth=2,
    padding=10,
    cornerRadius=4,
    fillColor='#ffffff',
    strokeColor='#dddddd'
).configure_view(
    strokeWidth=0
)

panel.save("line_panel.png")
print("Saved: line_panel.png")

Saved: line_panel.png
